**Setup**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/cv-longtail"

import sys
sys.path.append(ROOT)

import torch
import torch.nn.functional as F
print(torch.__version__)

Mounted at /content/drive
2.14.1+cpu


**Cross-Entropy**

In [ ]:
def ce_loss(logits, targets):
    log_probs = F.log_softmax(logits, dim=1)                  # log of the probability of each class
    picked = log_probs[torch.arange(len(targets)), targets]   # log-prob of the TRUE class, per image
    return -picked.mean()                                     # average over the batch, flip the sign

In [ ]:
# Test 1: if the model has no preference between 3 classes, loss should be log(3)
logits = torch.zeros(4, 3)
targets = torch.tensor([0, 1, 2, 0])
print(ce_loss(logits, targets), "expected about 1.0986")

# Test 2: should match PyTorch's built-in version
logits = torch.randn(8, 10)
targets = torch.randint(0, 10, (8,))
print(ce_loss(logits, targets), F.cross_entropy(logits, targets))
print(torch.allclose(ce_loss(logits, targets), F.cross_entropy(logits, targets)))

tensor(1.0986) expected about 1.0986
tensor(2.2430) tensor(2.2430)
True


**Class Reweighted Cross-Entropy**

In [ ]:
def inverse_freq_weights(class_counts):
    counts = torch.tensor(class_counts, dtype=torch.float)
    w = 1.0 / counts
    return w / w.sum() * len(counts)      # rescale so the weights average to 1

weights = inverse_freq_weights([5000, 2997, 1796, 1077, 645, 387, 232, 139, 83, 50])
print(weights)
print(weights.mean())    # should be 1.0

def weighted_ce_loss(logits, targets, weights):
    log_probs = F.log_softmax(logits, dim=1)
    picked = log_probs[torch.arange(len(targets)), targets]   # log-prob of true class
    w = weights.to(logits.device)[targets]                    # weight of each image's class
    return -(w * picked).sum() / w.sum()                      # weighted average

tensor([0.0402, 0.0671, 0.1120, 0.1868, 0.3119, 0.5199, 0.8672, 1.4474, 2.4239,
        4.0237])
tensor(1.)


In [ ]:
logits = torch.randn(8, 10)
targets = torch.randint(0, 10, (8,))

# Test 1: equal weights should give plain CE
ones = torch.ones(10)
print(torch.allclose(weighted_ce_loss(logits, targets, ones), ce_loss(logits, targets)))

# Test 2: should match PyTorch's built-in weighted CE
print(torch.allclose(weighted_ce_loss(logits, targets, weights),
                     F.cross_entropy(logits, targets, weight=weights)))

True
True


In [ ]:
from functools import partial

reweighted_ce = partial(weighted_ce_loss, weights=weights)

In [ ]:
logits = torch.randn(8, 10)
targets = torch.randint(0, 10, (8,))
print(reweighted_ce(logits, targets))

tensor(2.3491)


**Focal Loss**

In [ ]:
def focal_loss(logits, targets, gamma=2.0):
    log_probs = F.log_softmax(logits, dim=1)
    log_pt = log_probs[torch.arange(len(targets)), targets]   # log p_t, per image
    pt = log_pt.exp()                                         # p_t
    return (-(1 - pt) ** gamma * log_pt).mean()

In [ ]:
logits = torch.randn(8, 10)
targets = torch.randint(0, 10, (8,))

# Test 1: gamma=0 should give plain CE
print(torch.allclose(focal_loss(logits, targets, gamma=0.0), ce_loss(logits, targets)))

# Test 2: focal loss should never be larger than CE (the factor is between 0 and 1)
print(focal_loss(logits, targets).item() <= ce_loss(logits, targets).item())

# Test 3: check by hand on one image
logits = torch.tensor([[2.0, 0.0, 0.0]])
targets = torch.tensor([0])
pt = F.softmax(logits, dim=1)[0, 0].item()               # about 0.787
print(focal_loss(logits, targets).item(), -(1 - pt) ** 2 * torch.log(torch.tensor(pt)).item())

True
True
0.010869339108467102 0.010869338476755405


**Class-Balanced Loss**

In [ ]:
def cb_weights(class_counts, beta=0.9999):
    counts = torch.tensor(class_counts, dtype=torch.float)
    effective_num = (1.0 - beta ** counts) / (1.0 - beta)
    w = 1.0 / effective_num
    return w / w.sum() * len(counts)      # rescale so the weights average to 1

counts = [5000, 2997, 1796, 1077, 645, 387, 232, 139, 83, 50]
for beta in [0.9, 0.99, 0.999, 0.9999]:
    print(beta, cb_weights(counts, beta))

0.9 tensor([0.9995, 0.9995, 0.9995, 0.9995, 0.9995, 0.9995, 0.9995, 0.9995, 0.9996,
        1.0046])
0.99 tensor([0.7838, 0.7838, 0.7838, 0.7838, 0.7850, 0.8002, 0.8682, 1.0414, 1.3854,
        1.9844])
0.999 tensor([0.1812, 0.1894, 0.2157, 0.2729, 0.3785, 0.5606, 0.8688, 1.3862, 2.2585,
        3.6883])
0.9999 tensor([0.0506, 0.0769, 0.1211, 0.1950, 0.3188, 0.5246, 0.8684, 1.4426, 2.4092,
        3.9927])


In [ ]:
cb_ce = partial(weighted_ce_loss, weights=cb_weights(counts, beta=0.9999))

In [ ]:
logits = torch.randn(8, 10)
targets = torch.randint(0, 10, (8,))

# Test 1: weights average to 1
print(cb_weights(counts, 0.9999).mean())

# Test 2: with a tiny n, E_n should be close to n (beta=0.99, n=1 -> exactly 1)
print((1 - 0.99 ** 1) / (1 - 0.99))     # expect 1.0

# Test 3: rare classes must get larger weights than common ones
w = cb_weights(counts, 0.9999)
print(w[9] > w[0])

# Test 4: wrapped loss fits the interface (one scalar)
print(cb_ce(logits, targets))

tensor(1.)
1.0
tensor(True)
tensor(2.9263)


**LDAM-DRW**

In [ ]:
class LDAMLoss:
    def __init__(self, class_counts, max_m=0.5, s=30.0, drw_start=None, beta=0.9999):
        counts = torch.tensor(class_counts, dtype=torch.float)
        m = 1.0 / counts ** 0.25
        self.m = m * (max_m / m.max())          # rarest class gets max_m
        self.s = s
        self.drw_start = drw_start               # epoch to switch weights on (None = never)
        self.drw_weights = cb_weights(class_counts, beta)
        self.weights = None                      # no class weights at the start

    def set_epoch(self, epoch):                  # the training loop calls this once per epoch
        on = self.drw_start is not None and epoch >= self.drw_start
        self.weights = self.drw_weights if on else None

    def __call__(self, logits, targets):
        m = self.m.to(logits.device)[targets]                       # margin of each image's class
        onehot = F.one_hot(targets, logits.shape[1]).bool()
        adjusted = torch.where(onehot, logits - m[:, None], logits) # subtract margin from true class only
        w = None if self.weights is None else self.weights.to(logits.device)
        return F.cross_entropy(self.s * adjusted, targets, weight=w)

In [ ]:
counts = [5000, 2997, 1796, 1077, 645, 387, 232, 139, 83, 50]
logits = torch.rand(8, 10) * 2 - 1          # cosine-like values in [-1, 1]
targets = torch.randint(0, 10, (8,))

# Test 1: margins grow toward rare classes, max is 0.5
ldam = LDAMLoss(counts, drw_start=160)
print(ldam.m)                                # class 0 about 0.158, class 9 = 0.5

# Test 2: no margin, s=1, no DRW -> plain CE
print(torch.allclose(LDAMLoss(counts, max_m=0.0, s=1.0)(logits, targets), ce_loss(logits, targets)))

# Test 3: a margin can only make the loss bigger
print(LDAMLoss(counts, max_m=0.5)(logits, targets).item()
      >= LDAMLoss(counts, max_m=0.0)(logits, targets).item())

# Test 4: DRW switches on at epoch 160
ldam.set_epoch(0);   a = ldam(logits, targets); print(ldam.weights is None)
ldam.set_epoch(160); b = ldam(logits, targets); print(ldam.weights is not None, a.item() != b.item())

tensor([0.1581, 0.1797, 0.2042, 0.2321, 0.2638, 0.2998, 0.3407, 0.3872, 0.4405,
        0.5000])
True
True
True
True True
